# Example: Todo App

The main element of the app is a **task widget**. We will extend our previous [simple implementation](/topics/apps/01-flet.html#composite-controls) of this into a full, feature-rich application. The general idea is the same, every interactive control calls a hook that affects the app state and consequently the UI elements (e.g. visibility, focus) which we build imperatively.

## Initial version

Start by initializing the app with `uv run flet create`. This gives us an initial template:

```bash
$ tree .
.
├── README.md
├── pyproject.toml
└── src
    ├── assets
    │   ├── icon.png
    │   └── splash_android.png
    └── main.py

3 directories, 5 files
```

In a real project, you will first have to edit details in the `pyproject.toml` file and `README.md`. For our purposes, we will focus on the `main.py` file. We add the following code:

```{.python filename=src/v1.py}
import flet as ft

def main(page: ft.Page):
    def add_clicked(e):
        task_list.controls.append(ft.Checkbox(label=new_task.value))
        new_task.value = ""     # blank = show hint text again
        main_col.update()

    new_task = ft.TextField(
        hint_text="What needs to be done?", 
        expand=True, 
        on_submit=add_clicked   # ENTER triggers on_submit
    )
    add_button = ft.FloatingActionButton(
        icon=ft.Icons.ADD, 
        on_click=add_clicked
    )
    
    new_task_row = ft.Row(controls=[new_task, add_button])
    task_list = ft.Column()
    main_col = ft.Column(width=600, controls=[new_task_row, task_list])
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(main_col)


if __name__ == "__main__":
    ft.run(main)
```

Reading this backwards, we see that the main control is `main_col` which is centered horizontally. The main column contains the text field for adding a task followed by the task list. The task list is a column whose controls are appended with new task items that are implemented as **checkbox**.

<video
  src="./img/flet-todo/todo-v1.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

The main hook here is `add_clicked` on the ADD button. Moreover, pressing ENTER on the keyboard triggers the same hook. This appends a checkbox to the task list column, resets the text field to blank (which makes the text hint show), and finally updates the main control.

## Refactoring into a control subclass

Note that the app packages nicely as a single composite control with methods (e.g. the hooks). Moreover the app manages its data as class attributes. The following behaves exactly as the first one:

```{.python filename=src/v2.py}
import flet as ft

class TodoApp(ft.Column):
    def __init__(self, width: int):
        super().__init__(width=width)
        self.new_task = ft.TextField(
            hint_text="What needs to be done?", 
            expand=True, 
            on_submit=self.add_clicked   # ENTER triggers on_submit
        )
        self.add_button = ft.FloatingActionButton(
            icon=ft.Icons.ADD, 
            on_click=self.add_clicked
        )
        self.new_task_row = ft.Row(controls=[self.new_task, self.add_button])
        self.task_list = ft.Column()
        self.controls.extend([self.new_task_row, self.task_list])

    def add_clicked(self, e):
        self.task_list.controls.append(ft.Checkbox(label=self.new_task.value))
        self.new_task.value = ""    # blank = show hint text again
        self.update()


def main(page: ft.Page):
    todo = TodoApp(width=600)
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(todo)


if __name__ == "__main__":
    ft.run(main)
```

This feels like a cleaner more maintainable implementation of the original app.

## Task items as composite controls

Each **task item** is a composite column control consisting of two views. The default visible view is `display_view` which is a row consisting of three controls: (1) a checkbox with the **task name** (`.label`) and a clickable **toggle** (the state can be accessed via the a boolean attribute `.value`), (2) an edit button which hooks the `.edit_clicked(e)` method, and (3) a delete button which hooks the `.delete_clicked(e)` method. Next, we have the `edit_view` which is similarly a row with `TextField` and a save button which hooks `.save_clicked(e)`. 

![](./img/flet-todo/task_item.drawio.png)

^Actually the checkbox doesn't extend as in the figure. Otherwise, the ff. demo shows the buttons behavior:

<video
  src="./img/flet-todo/todo-v3.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

The definition of each hook can be seen in the code:

```{.python filename=src/v3.py}
import flet as ft
from typing import Callable

class TaskItem(ft.Column):
    def __init__(self, text: str, delete_hook: Callable):
        super().__init__()
        self.delete_hook = delete_hook
        self.checkbox = ft.Checkbox(label=text)
        
        # buttons
        self.edit_button = ft.IconButton(
            icon=ft.Icons.EDIT,
            on_click=self.edit_clicked
        )

        self.delete_button = ft.IconButton(
            icon=ft.Icons.DELETE,
            on_click=self.delete_clicked
        )

        self.save_button = ft.IconButton(
            icon=ft.Icons.SAVE,
            on_click=self.save_clicked
        )

        # two views
        self.display_view = ft.Row(controls=[
            self.checkbox, self.edit_button, self.delete_button
        ])

        self.edit_view = ft.Row(
            controls=[
                ft.TextField(
                    value=self.checkbox.label, 
                    expand=True, 
                    on_submit=self.save_clicked
                ),
                self.save_button,
            ], 
            visible=False
        )

        self.controls.extend([self.display_view, self.edit_view])

    def delete_clicked(self, e):
        """Remove this task from the todo list using external hook."""
        self.delete_hook(self)

    def edit_clicked(self, e):
        self.display_view.visible = False
        self.edit_view.visible = True
        self.update()

    def save_clicked(self, e):
        self.checkbox.label = self.edit_view.controls[0].value
        self.display_view.visible = True
        self.edit_view.visible = False
        self.update()

    def is_isolated(self):
        return True
...
```

Observe that the hooks explicitly flips the visibility of the two views so that only one view is visible at each time. Clicking edit, makes only the `edit_view` visible. This makes the edit field visible and clicking save does the reverse while replacing the checkbox label to the contents of the `TextField`. Finally, delete task signals the delete hook which calls an [external]{.underline} function which makes sense since its the outer app container which handles the list of `TaskItem`. Let us now proceed with the `TodoApp` which maintains this said list:

```{.python filename=src/v3.py}
...

class TodoApp(ft.Column):
    def __init__(self, page: ft.Page, width: int):
        super().__init__(width=width)
        self._page = page
        self.new_task = ft.TextField(
            hint_text="What needs to be done?", 
            expand=True, 
            on_submit=self.add_clicked   # ENTER triggers on_submit
        )
        self.add_button = ft.FloatingActionButton(
            icon=ft.Icons.ADD, 
            on_click=self.add_clicked
        )
        self.task_list = ft.Column()
        self.controls.extend([
            ft.Row(controls=[self.new_task, self.add_button]), 
            self.task_list
        ])

    def add_clicked(self, e):
        task = TaskItem(self.new_task.value, delete_hook=self.delete_task)
        self.task_list.controls.append(task)
        self.new_task.value = ""    # blank = show hint text again
        self.update()

    def is_isolated(self):
        return True
    
    def delete_task(self, task: TaskItem):
        dlg_modal = ft.AlertDialog(
            modal=True,
            title=ft.Text("Confirm delete"),
            content=ft.Text("Are you sure you want to delete this task?"),
            actions=[
                ft.Button(
                    "Yes", 
                    on_click=lambda e: (
                        self.task_list.controls.remove(task), 
                        self.update(), 
                        self._page.pop_dialog()
                    )
                ),
                ft.TextButton(
                    "No", 
                    on_click=lambda e: self._page.pop_dialog()
                ),
            ],
            actions_alignment=ft.MainAxisAlignment.END,
        )
        self._page.show_dialog(dlg_modal)
        self._page.update()
        

def main(page: ft.Page):
    todo = TodoApp(page, width=600)
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(ft.Text("Todo list 📝", size=50, weight=ft.FontWeight.BOLD), todo)

if __name__ == "__main__":
    ft.run(main)
```

You can see that the final app is simply the previous version a column containing a text field for new tasks and a column containing the task items. Hence, in this version, we added the task item functionality. 

Additionally, we have the `delete_task` function which opens a **delete modal** and removes a task from the list. Note that `.remove` is $O(n)$ but since the task list is <100 or <1000 at the very extreme (probably this has to be enforced in an actual app), it should be fine. The `delete_task` function is inserted into each `TaskItem` which calls `delete_task(self)` allowing the main `TodoApp` to know which task to delete from its list.

:::{.callout-note}
Here we used `TextButton` for "No" and `Button` for "Yes" in the delete modal. However, it's not apparent from the code that we only selected one over the other because of **styling** reasons &mdash; the latter is elevated while the former blends into the background. It might be better to make styling explicit and just use `Button`.
:::

## Final enhancements: active tasks, tab filters, focus

This final section from a UX perspective adds quality of life improvements to the application. But from the developer's point of view, the current code change versus the last section reflects how complex tracking state is with the imperative approach. In the next notebook, we will see how to implement this in a **declarative** way which may be a more sane way to handle state for complex applications.

<video
  src="./img/flet-todo/todo-v4.mov"
  autoplay
  loop
  muted
  controlslist="nodownload"
  oncontextmenu="return false"
  style="max-width:100%;">
</video>

There is a lot to unpack here. First, we see that the app does **autofocus** on text fields, e.g. the new task text field when the app starts, and immediately when a new task is added. Moreover, it focuses on the text field during task item edit mode. Also, a scrollbar appears whenever the allotted vertical space is exceeded by the task list and **autoscrolls** whenever a new task is added. We also see a footer which prints the number of active tasks. This is decreased whenever a task item is toggled as completed. 

The **tab filters** allow us to view task items with active or completed statuses. Note that any change while in one of the tabs applies to when we're at other tabs. This seems trivial, but in an imperative approach, making sure the state is consistent along any trajectory into the application is something that you may have to spend a lot of time on. Finally, the clear completed button removes all completed tasks. This means the active count must not be affected which can be seen here.

```{.python filename=src/v4.py}
import flet as ft
from typing import Callable

class TaskItem(ft.Column):
    def __init__(self, text: str, status_hook: Callable, delete_hook: Callable):
        super().__init__()
        
        self.delete_hook = delete_hook
        self.checkbox = ft.Checkbox(label=text, on_change=status_hook)
        self.edit_button = ft.IconButton(
            icon=ft.Icons.EDIT,
            on_click=self.edit_clicked
        )

        self.delete_button = ft.IconButton(
            icon=ft.Icons.DELETE,
            on_click=self.delete_clicked
        )

        self.save_button = ft.IconButton(
            icon=ft.Icons.SAVE,
            on_click=self.save_clicked
        )

        self.display_view = ft.Row(controls=[
            self.checkbox, self.edit_button, self.delete_button
        ])

        self.edit_field = ft.TextField(
            value=self.checkbox.label, 
            expand=True, 
            on_submit=self.save_clicked
        )

        self.edit_view = ft.Row(
            controls=[
                self.edit_field,
                self.save_button,
            ], 
            visible=False
        )

        self.controls.extend([self.display_view, self.edit_view])

    def delete_clicked(self, e):
        """Remove this task from the todo list using external hook."""
        self.delete_hook(self)

    async def edit_clicked(self, e):
        self.display_view.visible = False
        self.edit_view.visible = True
        self.update()
        await self.edit_field.focus()

    def save_clicked(self, e):
        self.checkbox.label = self.edit_field.value
        self.display_view.visible = True
        self.edit_view.visible = False
        self.update()

    def is_isolated(self):
        return True
...
```

What is added here is a `status_hook` for when the task status changes and we're monitoring active count and accurate views within tab filters. Note that forcing updates is necessary since each `TaskItem` is [isolated](/topics/apps/01-flet.html#isolated-controls). Next, the `edit_clicked` function is now async which is needed for **focus**. OK, that's fine. Now let us look at the main app: 

```{.python filename=src/v4.py}
...

class TodoApp(ft.Column):
    TAB_ALL = "all"
    TAB_ACTIVE = "active"
    TAB_COMPLETED = "completed"
    
    def __init__(self, page: ft.Page, width: int):
        super().__init__(width=width)
        self._page = page
        
        self.new_task = ft.TextField(
            hint_text="What needs to be done?", 
            expand=True, 
            on_submit=self.add_clicked   # ENTER triggers on_submit
        )
        
        self.add_button = ft.FloatingActionButton(
            icon=ft.Icons.ADD, 
            on_click=self.add_clicked
        )
        
        self.task_list = ft.ListView(height=250, spacing=10, auto_scroll=True)

        self.filter = ft.Tabs(
            selected_index=0,
            length=3,
            on_change=self.tabs_changed,
            content=ft.TabBar(
                scrollable=False,
                tabs=[
                    ft.Tab(label=TodoApp.TAB_ALL), 
                    ft.Tab(label=TodoApp.TAB_ACTIVE), 
                    ft.Tab(label=TodoApp.TAB_COMPLETED)
                ],
            )
        )

        self.active_count = ft.Text(
            "0 active tasks left.", 
            color=ft.Colors.GREY_400
        )

        self.clear_completed = ft.Button(
            "Clear Completed", 
            on_click=lambda e: self.confirm_clear_completed(),
            style=ft.ButtonStyle(shape=ft.RoundedRectangleBorder(radius=10))
        )

        self.controls.extend([
            ft.Row(controls=[self.new_task, self.add_button]), 
            self.filter,
            self.task_list,
            ft.Divider(thickness=0.5, color=ft.Colors.GREY_600),
            ft.Row(
                controls=[self.active_count, self.clear_completed], 
                alignment=ft.MainAxisAlignment.SPACE_BETWEEN
            )
        ])

    async def add_clicked(self, e):
        task = TaskItem(
            self.new_task.value, 
            status_hook=self.on_status_change, 
            delete_hook=self.confirm_delete_task
        )
        self.task_list.controls.append(task)
        self.new_task.value = ""    # note: blank = show hint text again
        self.on_status_change()     # new task => reflect +1 to active count
        self.update()
        await self.new_task.focus()

    def is_isolated(self):
        return True
    
    def on_status_change(self):
        num_active = sum(1 for task in self.task_list.controls if not self.is_completed(task))
        self.active_count.value = f"{num_active} active tasks left."
        self.update()

    def confirm_delete_task(self, task):
        delete_dialog = ft.AlertDialog(
            modal=True,
            title=ft.Text("Confirm delete"),
            content=ft.Text("Are you sure you want to delete this task?"),
            actions=[
                ft.Button(
                    "Yes", 
                    on_click=lambda e: (
                        self.delete_task(task), 
                        self.update(), 
                        self._page.pop_dialog()
                    )
                ),
                ft.TextButton(
                    "No", 
                    on_click=lambda e: self._page.pop_dialog()
                ),
            ],
            on_dismiss=lambda e: self.on_status_change(),
            actions_alignment=ft.MainAxisAlignment.END,
        )
        self._page.show_dialog(delete_dialog)
        self._page.update()
    
    def delete_task(self, task: TaskItem):
        self.task_list.controls.remove(task)
        
    def confirm_clear_completed(self):
        delete_dialog = ft.AlertDialog(
            modal=True,
            title=ft.Text("Confirm delete"),
            content=ft.Text("Are you sure you want to delete completed tasks?"),
            actions=[
                ft.Button(
                    "Yes", 
                    on_click=lambda e: (
                        self.delete_completed_tasks(),
                        self.update(), 
                        self._page.pop_dialog()
                    )
                ),
                ft.TextButton(
                    "No", 
                    on_click=lambda e: self._page.pop_dialog()
                ),
            ],
            on_dismiss=lambda e: self.on_status_change(),
            actions_alignment=ft.MainAxisAlignment.END,
        )
        self._page.show_dialog(delete_dialog)
        self._page.update()

    def delete_completed_tasks(self):
        for task in self.task_list.controls[:]:
            if self.is_completed(task):
                self.task_list.controls.remove(task)
    
    def tabs_changed(self, e):
        self.update()

    def is_completed(self, task: TaskItem):
        return task.checkbox.value

    def before_update(self):
        visible_fn = {
            TodoApp.TAB_ALL: lambda task: True,
            TodoApp.TAB_ACTIVE: lambda task: not self.is_completed(task),
            TodoApp.TAB_COMPLETED: lambda task: self.is_completed(task),
        }
        selected_idx = self.filter.selected_index
        selected_tab = self.filter.content.tabs[selected_idx].label
        for task in self.task_list.controls:
            task.visible = visible_fn[selected_tab](task)


async def main(page: ft.Page):
    todo = TodoApp(page, width=600)
    page.horizontal_alignment = ft.CrossAxisAlignment.CENTER
    page.add(ft.Text("Todo list 📝", size=50, weight=ft.FontWeight.BOLD), todo)
    await todo.new_task.focus()

if __name__ == "__main__":
    ft.run(main)
```

Again reading backward, we see that the app starts by focusing on the `new_task` field. This is refocused each time a new task is added as can be seen in the `await self.new_task.focus()` line. The `todo` app is a composite column control that consists of the new task field, followed by the tab filters, the task list, and the footer. Note that the add task hook `add_clicked` triggers `self.on_status_change()` which rebuilds the footer with the new count (i.e. +1):

```python
def on_status_change(self):
    num_active = sum(1 for task in self.task_list.controls if not self.is_completed(task))
    self.active_count.value = f"{num_active} active tasks left."
    self.update()
```

Again, the new task field is a row consisting of a `TextField` and a `FloatingActionButton` with an ADD icon. Autoscroll for the task list turns out to be implemented in the Flet library, hence we simply do:

```python
self.task_list = ft.ListView(height=250, spacing=10, auto_scroll=True)
```

Managing the behavior w.r.t. tab filters is a bit more involved. First, it is defined as follows:

```python
TAB_ALL = "all"
TAB_ACTIVE = "active"
TAB_COMPLETED = "completed"

self.filter = ft.Tabs(
    selected_index=0,
    length=3,
    on_change=self.tabs_changed,
    content=ft.TabBar(
        scrollable=False,
        tabs=[
            ft.Tab(label=TodoApp.TAB_ALL), 
            ft.Tab(label=TodoApp.TAB_ACTIVE), 
            ft.Tab(label=TodoApp.TAB_COMPLETED)
        ],
    )
)
```

Hence, it starts with "all", then "active" in the middle, and "completed" last. To actually trigger the effect of these in the UI, a hook is triggered `self.tabs_changed` which actually only does a `.update()` of the `todo` app. The trick is to use [lifecycle methods](https://docs.flet.dev/cookbook/custom-controls/#life-cycle-methods). Notice that by doing CTRL+F, you will not find a usage of `before_update`. This turns out to be a lifecycle method that is run each time the component is updated. 

Since `self.update()` is performed within `tabs_changed` in `todo` when a new tab is selected, this then runs `before_update`. Moreover, this is done whenever any other part of the app updates (e.g. a task item toggles status, then `on_status_change` is triggered which does an update, so that `before_update` is also inserted as an in between). The lifecycle method simply updates the visibility of the tasks depending on the tab selected:

```python
def before_update(self):    # lifecycle method! self = todo
    visible_fn = {
        TodoApp.TAB_ALL: lambda task: True,
        TodoApp.TAB_ACTIVE: lambda task: not self.is_completed(task),
        TodoApp.TAB_COMPLETED: lambda task: self.is_completed(task),
    }
    selected_idx = self.filter.selected_index
    selected_tab = self.filter.content.tabs[selected_idx].label
    for task in self.task_list.controls:
        task.visible = visible_fn[selected_tab](task)
```

:::{.callout-warning}
`before_update()` method is called every time when the control is being updated. Make sure not to call `update()` method within `before_update()`.
:::

Finally, we have a button for clearing completed tasks. This simply renders a dialog modal that calls a `delete_completed_tasks` which iterates over completed tasks and removes them from the task list. A subtle detail is that we iterate over a copy `self.task_list.controls[:]` since removing modifies the list (little Python gotcha).

## Final remarks 

The final code looks clear and easy to implement. But during development, it took a lot of trial and error to figure out, for example, the minimal number of hooks and the simplest usage to maintain the consistency between UI and actual task status. Also, it also takes mental load to decide where to put page updates. 

Of course, referring to actual **application state** (e.g. counting completed tasks at each update instead of using proxies like a counter) is the better approach. But the imperative model does not emphasize or streamline this natively. In the next notebook we look at a way to develop Flet apps that treats state and UI components as separate with state as being the single source of truth which is reflected in the UI.